In [1]:
import sys
import json
from tqdm import tqdm

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.qa_pipeline import QAPipeline
from src.qa_pipeline.query_parser import QueryLLMParser
from src.qa_pipeline.knowledge_comparator import KnowledgeComparator
from src.qa_pipeline.knowledge_retriever import KnowledgeRetriever
from src.qa_pipeline.answer_generator import QALLMGenerator

from src.llm_agent import AgentConnector
from src.knowledge_graph_model import KnowledgeGraphModel
from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection

from src.embedding_functions import ChromaConnection, VectorDBConnectionConfig, EmbeddingsDatabaseConnectionConfig

In [2]:
agent = AgentConnector.open()
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri="bolt://31.207.47.254:7687", user="neo4j", pwd="password", db_name="testdb"),
    embeddings_db=EmbeddingsDatabaseConnection()
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with MEAN pooling.


In [3]:
QUERY = "Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?"

### QA-пайплан целиком

In [4]:
qa_pipeline = QAPipeline(kg_model, agent)
qa_pipeline.answer(QUERY)

'Xiaomi 11'

### QA-пайплайн по частям

In [5]:
# stage 1
q_parser = QueryLLMParser(agent)
qparser_out = q_parser.extract_entities(QUERY)
print(qparser_out.entities)

['device', 'battery life', 'iPhone11 Pro Max', 'Xiaomi 11']


In [6]:
# stage 2
k_comparator = KnowledgeComparator(kg_model)
k_comparator.link_kgnodes_to_query(qparser_out)

for item in qparser_out.linked_nodes:
    print(item)

VectorDBInstance(id='4:d958299b-8cff-4454-876f-4f337d0518bd:1', document='video', embedding=[0.03109743259847164, 0.0004729302891064435, -0.05565173178911209, -0.06460603326559067, 0.07972336560487747, 0.04102066904306412, 0.03337995335459709, 0.024896876886487007, 0.05651523172855377, -0.005306700710207224, 0.040258102118968964, -0.012111412361264229, 0.07159245014190674, -0.05147212743759155, -0.03100411221385002, 0.0195023063570261, 0.0578431710600853, 0.01876750774681568, -0.05561463162302971, -0.02937053143978119, 0.036580007523298264, -0.015255092643201351, -0.021429816260933876, 0.02005075104534626, 0.06712803989648819, 0.032416798174381256, -0.03720080479979515, -0.025249294936656952, 0.048969488590955734, -0.06928988546133041, -0.03220387548208237, -0.03981589525938034, 0.06511982530355453, -0.05428413301706314, 0.057315126061439514, 0.057197682559490204, -0.016292797401547432, -0.034584611654281616, 0.052730388939380646, -0.013432648964226246, -0.013661707751452923, 0.0307748

In [7]:
# stage 3
k_retriever = KnowledgeRetriever(kg_model)
kretriever_out = k_retriever.retrieve(qparser_out)

for item in kretriever_out:
    print(item)

Triplet(start_node=Node(name='xiaomi_mi_11', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:126', prop={'kind': 'device', 'name': 'xiaomi_mi_11'}), relation=Relation(name='opinion', type=<RelationType.simple: 'simple'>, id='5:d958299b-8cff-4454-876f-4f337d0518bd:1841', prop={'raw_time': '25165', 'sentiment': 'neg', 'person': 'Bernard', 'name': 'opinion', 'time': '15.11.2020', 'opinion': 'poor'}), end_node=Node(name='battery_life', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:17', prop={'kind': 'feature', 'name': 'battery_life'}))
Triplet(start_node=Node(name='xiaomi_mi_11', type=<NodeType.object: 'object'>, id='4:d958299b-8cff-4454-876f-4f337d0518bd:126', prop={'kind': 'device', 'name': 'xiaomi_mi_11'}), relation=Relation(name='opinion', type=<RelationType.simple: 'simple'>, id='5:d958299b-8cff-4454-876f-4f337d0518bd:4806', prop={'raw_time': '18836', 'sentiment': 'neu', 'person': 'Lewis', 'name': 'opinion', 'time': '24.2.202

In [8]:
# stage 4
qa_generator = QALLMGenerator(agent)
contexts = qa_generator.formate_context(kretriever_out)
qagenerator_out = qa_generator.generate(qparser_out.query, contexts)

print(contexts)
print()
print(qagenerator_out)

15.11.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: neg; person: Bernard; time: 15.11.2020; opinion: poor) battery_life (kind: feature)
24.2.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: neu; person: Lewis; time: 24.2.2020; opinion: its_just_okay) battery_life (kind: feature)
28.12.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: pos; person: Bernard; time: 28.12.2020; opinion: its_incredible) battery_life (kind: feature)
6.12.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: pos; person: Bernard; time: 6.12.2020; opinion: incredible_battery_life) battery_life (kind: feature)
17.10.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: neg; person: Melanie; time: 17.10.2020; opinion: always_so_shaky) video (kind: feature)
24.2.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: pos; person: Sophia; time: 24.2.2020; opinion: insane_video_quality) video (kind: feature)
30.11.2020: xiaomi_mi_11 (kind: device) opinion (sentiment: neg; person: Melanie; time: 30.11.